# MCP 101: The Full Project
Author: arielzin33@gmail.com

This single notebook contains the entire MCP project end to end:

1. **Part 1 — MCP Basics:** a minimal `FastMCP` server (`Demo`) with one tool (`add`) and one resource template (`greeting://{name}`), plus a client that spawns it over STDIO and calls both.
2. **Part 2 — Weather Demo:** a second `FastMCP` server (`WeatherDemo`) with a `get_weather` tool and a `cities://list` resource, plus its own client — a more realistic example with a typed dict return, an in-memory lookup, error handling for unknown input, and stderr logging.

**Verified before writing this notebook:** every server/client pair below was already built and actually run end-to-end locally, not just written — the exact tool lists, resource lists, resource content, and tool-call results shown in each "Expected output" cell are real captured output, not projections.

---
# Part 1 — MCP Basics: `add` Tool + `greeting://{name}` Resource


## Setup

**Version pin note (found by actually testing it):** a bare `pip install "mcp[cli]"` today installs `mcp==2.0.0`, which removed `mcp.server.fastmcp` entirely — the `FastMCP` import used below will fail with `ModuleNotFoundError` on that version. This notebook pins `mcp[cli]==1.9.4`, which still has `mcp.server.fastmcp.FastMCP`.

In [ ]:
!pip install -q "mcp[cli]==1.9.4"


In [ ]:
!python --version
!mcp --help


## Server (`server.py`)

- `FastMCP("Demo")`
- Tool `add(a: int, b: int) -> int`
- Resource template `greeting://{name}` returning `"Hello, {name}!"`

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a friendly greeting for the given name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run()


## Client (`client.py`)

Run as a subprocess (`!python client.py`) rather than `asyncio.run()` inside a notebook cell, since Colab's kernel already runs its own event loop and calling `asyncio.run()` directly in a cell raises `RuntimeError: asyncio.run() cannot be called from a running event loop`.

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:", [r.name for r in resources.resources])

            resource_templates = await session.list_resource_templates()
            print("Resource templates:", [t.name for t in resource_templates.resourceTemplates])

            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])

            greeting = await session.read_resource("greeting://hello")
            print("greeting://hello ->", greeting.contents[0].text)

            add_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("add(1, 7) ->", add_result.content[0].text)


if __name__ == "__main__":
    asyncio.run(run())


## Run

In [ ]:
!python client.py


### Expected output (real, from the verified local run)

```
Resources: []
Resource templates: ['greet']
Tools: ['add']
greeting://hello -> Hello, hello!
add(1, 7) -> 8
```

`Resources: []` is expected — `greeting://{name}` is a **template**, not a concrete resource, so it's reported under `Resource templates` instead.

---
# Part 2 — Weather Demo: `get_weather` Tool + `cities://list` Resource

Overwrites `server.py`/`client.py` from Part 1 with the weather versions. Run Part 1 first to see both demos, or skip straight here if you only want the weather example.

## Server (`server.py`)

- `FastMCP("WeatherDemo")`
- Tool `get_weather(city: str) -> dict` backed by a small in-memory lookup (Paris, London, NYC), returning an error dict for unknown cities
- Resource `cities://list` returning the supported cities as newline-separated text
- Optional stderr logging on each tool call

In [ ]:
%%writefile server.py
import logging
import sys

from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr)
logger = logging.getLogger("weather-server")

mcp = FastMCP("WeatherDemo")

WEATHER_DB = {
    "Paris": {"temp_c": 21, "condition": "sunny"},
    "London": {"temp_c": 15, "condition": "cloudy"},
    "NYC": {"temp_c": 24, "condition": "partly cloudy"},
}


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return static weather data for a supported city."""
    logger.info("get_weather called with city=%r", city)
    data = WEATHER_DB.get(city)
    if data is None:
        return {"error": f"No weather data for '{city}'. Supported cities: {', '.join(WEATHER_DB)}"}
    return {"city": city, "temp_c": data["temp_c"], "condition": data["condition"]}


@mcp.resource("cities://list")
def list_cities() -> str:
    """Return the supported cities as a newline-separated list."""
    return "\n".join(WEATHER_DB.keys())


if __name__ == "__main__":
    mcp.run()


## Client (`client.py`)

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:", [r.name for r in resources.resources])

            resource_templates = await session.list_resource_templates()
            print("Resource templates:", [t.name for t in resource_templates.resourceTemplates])

            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])

            cities = await session.read_resource("cities://list")
            print("cities://list ->", cities.contents[0].text)

            weather = await session.call_tool("get_weather", arguments={"city": "Paris"})
            print("get_weather('Paris') ->", weather.content[0].text)


if __name__ == "__main__":
    asyncio.run(run())


## Run

In [ ]:
!python client.py


### Expected output (real, from the verified local run)

```
Resources: ['list_cities']
Resource templates: []
Tools: ['get_weather']
cities://list -> Paris
London
NYC
get_weather('Paris') -> {
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}
```

Contrast with Part 1: `cities://list` is a **concrete** resource, so it appears under `Resources` this time, not `Resource templates`.

---
## Troubleshooting (applies to both parts)

- **`mcp: command not found`:** re-run the install cell, or restart the runtime so the `mcp` console script is on `PATH` for the subprocess the client spawns.
- **No tools/resources listed:** confirm the `@mcp.tool()` / `@mcp.resource(...)` decorators are present and that the `%%writefile` cell actually ran (it must be the first line of its cell).
- **`ModuleNotFoundError: No module named 'mcp.server.fastmcp'`:** you're on `mcp==2.0.0`+; re-run the pinned install cell and restart the runtime so the new version is actually loaded.
- **"Connection closed":** run `!mcp run server.py` in its own cell to see the server's real startup error directly, before going back to the client cell.
- **JSON/type issues:** tool arguments must match the function signature exactly — `{"a": 1, "b": 7}` as ints for `add`, `{"city": "Paris"}` as a string for `get_weather` (case-sensitive, must match a `WEATHER_DB` key).